In [0]:
%pip install xgboost scikit-learn
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Mesmo seed do notebook 40
SEED = 42
np.random.seed(SEED)

print("="*80)
print("🎯 AUDITORIA TOP 1 - DATAS NÃO SOBREPOSTAS (7 PREGÕES)")
print("="*80)
print(f"\nSeed: {SEED}")
print("\nObjetivo: Recalcular taxa Top 1 usando apenas datas espaçadas a cada 7 pregões")
print("Método: Preservar modelo, probabilidades e previsões do notebook 40")
print("="*80)

In [0]:
# Carregar Gold V1
df = spark.table("workspace.gold.fii_features_v1").toPandas()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f"✅ Gold V1 carregada: {df.shape[0]:,} registros")
print(f"Período: {df['date'].min().date()} a {df['date'].max().date()}")
print(f"Tickers: {sorted(df['ticker'].unique())}")

# Features (todas as colunas exceto ticker, date, target_7d, target_alpha_7d)
features = [
    'close', 'volume', 'has_trading',
    'return_1d', 'return_7d', 'return_30d', 'return_90d',
    'ifix_return_1d', 'ifix_return_7d', 'ifix_return_30d', 'ifix_return_90d',
    'volatility_30d', 'volatility_90d',
    'dividend_yield_12m', 'dividend_history_days', 'days_since_last_dividend', 'has_dividend_today',
    'alpha_30d', 'alpha_90d',
    'selic', 'ipca', 'dolar', 'desemprego'
]

print(f"\nFeatures: {len(features)}")

In [0]:
# Período de holdout (idêntico ao notebook 40)
holdout_start = pd.to_datetime('2025-03-01')
holdout_end = pd.to_datetime('2026-02-28')

# Encontrar última data de treino (7 pregões antes do holdout)
dates_before_holdout = sorted(df[df['date'] < holdout_start]['date'].unique())
last_train_date = dates_before_holdout[-8]  # -8 para garantir gap de 7 pregões

print(f"Período de holdout: {holdout_start.date()} a {holdout_end.date()}")
print(f"Última data de treino: {last_train_date.date()}")
print(f"Gap de purge: 7 pregões")

# Dividir treino/holdout
train_df = df[df['date'] <= last_train_date].copy()
holdout_df = df[(df['date'] >= holdout_start) & (df['date'] <= holdout_end)].copy()

print(f"\nTreino: {len(train_df):,} registros")
print(f"Holdout: {len(holdout_df):,} registros")

In [0]:
# Hiperparâmetros CONGELADOS (idênticos ao notebook 40)
FINAL_HYPERPARAMETERS = {
    'n_estimators': 150,
    'max_depth': 6,
    'learning_rate': 0.07,
    'min_child_weight': 1,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'gamma': 0.0,
    'reg_alpha': 0.01,
    'reg_lambda': 1.5,
    'early_stopping_rounds': 20,
    'random_state': SEED,
    'n_jobs': -1,
    'tree_method': 'hist'
}

print("🔧 Hiperparâmetros congelados (do notebook 40):")
for k, v in FINAL_HYPERPARAMETERS.items():
    print(f"  {k}: {v}")

In [0]:
# Preparar X, y
X_train = train_df[features].values
y_train = train_df['target_7d'].values
X_holdout = holdout_df[features].values
y_holdout = holdout_df['target_7d'].values

print(f"X_train: {X_train.shape}")
print(f"X_holdout: {X_holdout.shape}")

# Treinar modelo (idêntico ao notebook 40)
model = xgb.XGBClassifier(**FINAL_HYPERPARAMETERS)
model.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)

print(f"\n✅ Modelo treinado (seed={SEED})")

# Predições no holdout
y_pred_proba = model.predict_proba(X_holdout)[:, 1]

print(f"\nPredições: {len(y_pred_proba):,}")
print(f"Probabilidade média: {y_pred_proba.mean():.4f}")
print(f"Probabilidade min/max: {y_pred_proba.min():.4f} / {y_pred_proba.max():.4f}")

In [0]:
# Adicionar previsões ao holdout_df
holdout_df_eval = holdout_df.copy()
holdout_df_eval['y_pred_proba'] = y_pred_proba

print(f"✅ Previsões adicionadas ao holdout_df")
print(f"Colunas: {list(holdout_df_eval.columns)}")
print(f"\nPrimeiras 5 linhas:")
print(holdout_df_eval[['date', 'ticker', 'target_7d', 'target_alpha_7d', 'y_pred_proba']].head())

In [0]:
# Selecionar datas não sobrepostas espaçadas a cada 7 pregões
all_dates = sorted(holdout_df_eval['date'].unique())

print(f"Total de datas no holdout: {len(all_dates)}")
print(f"Primeira data: {all_dates[0].date()}")
print(f"Última data: {all_dates[-1].date()}")

# Começar na primeira data e avançar 7 pregões por vez
non_overlapping_dates = []
current_idx = 0

while current_idx < len(all_dates):
    non_overlapping_dates.append(all_dates[current_idx])
    current_idx += 7  # Avançar exatamente 7 pregões

print(f"\n✅ Datas não sobrepostas selecionadas: {len(non_overlapping_dates)}")
print(f"\nPrimeiras 5 datas:")
for i, d in enumerate(non_overlapping_dates[:5]):
    print(f"  {i+1}. {d.date()}")

print(f"\nÚltimas 5 datas:")
for i, d in enumerate(non_overlapping_dates[-5:], start=len(non_overlapping_dates)-4):
    print(f"  {i}. {d.date()}")

In [0]:
# Calcular Top 1 para cada data não sobreposta
top1_results_non_overlapping = []

for date in non_overlapping_dates:
    date_data = holdout_df_eval[holdout_df_eval['date'] == date].copy()
    
    # Ranquear por probabilidade (descendente)
    date_data = date_data.sort_values('y_pred_proba', ascending=False)
    
    if len(date_data) > 0:
        top1 = date_data.iloc[0]
        
        # Verificar se o Top 1 superou o IFIX
        top1_beat_ifix = top1['target_7d']
        
        # Verificar se foi o melhor retorno absoluto entre os 5 FIIs
        best_alpha = date_data['target_alpha_7d'].max()
        top1_alpha = top1['target_alpha_7d']
        top1_best_return = (top1_alpha == best_alpha)
        
        top1_results_non_overlapping.append({
            'date': date,
            'top1_ticker': top1['ticker'],
            'top1_proba': top1['y_pred_proba'],
            'top1_beat_ifix': top1_beat_ifix,
            'top1_best_return': top1_best_return,
            'top1_alpha_7d': top1_alpha,
            'best_alpha_7d': best_alpha
        })

top1_df_non_overlapping = pd.DataFrame(top1_results_non_overlapping)

print(f"✅ Análise Top 1 concluída para {len(top1_df_non_overlapping)} datas não sobrepostas")
print(f"\nPrimeiras 10 seleções:")
print(top1_df_non_overlapping[['date', 'top1_ticker', 'top1_proba', 'top1_beat_ifix', 'top1_best_return']].head(10))

In [0]:
# Estatísticas principais
total_dates_non_overlapping = len(top1_df_non_overlapping)
top1_hits_non_overlapping = top1_df_non_overlapping['top1_beat_ifix'].sum()
top1_rate_non_overlapping = top1_hits_non_overlapping / total_dates_non_overlapping

top1_best_hits_non_overlapping = top1_df_non_overlapping['top1_best_return'].sum()
top1_best_rate_non_overlapping = top1_best_hits_non_overlapping / total_dates_non_overlapping

print("="*80)
print("🏆 ESTATÍSTICAS TOP 1 - DATAS NÃO SOBREPOSTAS")
print("="*80)

print(f"\n📊 Números Principais:")
print(f"  Total de datas não sobrepostas: {total_dates_non_overlapping}")
print(f"  Acertos (Top 1 > IFIX): {top1_hits_non_overlapping}")
print(f"  Taxa Top 1: {top1_rate_non_overlapping:.4f} ({top1_rate_non_overlapping*100:.2f}%)")

print(f"\n🎯 Top 1 foi o Melhor FII Absoluto:")
print(f"  Vezes que foi o melhor: {top1_best_hits_non_overlapping}")
print(f"  Taxa: {top1_best_rate_non_overlapping:.4f} ({top1_best_rate_non_overlapping*100:.2f}%)")

# Intervalo de confiança de 95% (proporção binomial)
n = total_dates_non_overlapping
p_hat = top1_rate_non_overlapping
z = 1.96  # 95% confidence
se = np.sqrt(p_hat * (1 - p_hat) / n)
ci_lower = p_hat - z * se
ci_upper = p_hat + z * se

print(f"\n📊 Intervalo de Confiança de 95%:")
print(f"  IC 95%: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"  IC 95%: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print(f"  Margem de erro: ±{z*se*100:.2f} pontos percentuais")

In [0]:
# Taxa por ticker selecionado
print("="*80)
print("📊 TAXA POR TICKER SELECIONADO")
print("="*80)

ticker_stats_non_overlapping = []

for ticker in sorted(top1_df_non_overlapping['top1_ticker'].unique()):
    ticker_data = top1_df_non_overlapping[top1_df_non_overlapping['top1_ticker'] == ticker]
    
    total = len(ticker_data)
    hits = ticker_data['top1_beat_ifix'].sum()
    rate = hits / total if total > 0 else 0
    
    ticker_stats_non_overlapping.append({
        'ticker': ticker,
        'selecionado': total,
        'acertos': hits,
        'taxa': rate
    })

ticker_stats_df = pd.DataFrame(ticker_stats_non_overlapping)
ticker_stats_df = ticker_stats_df.sort_values('selecionado', ascending=False)

print(f"\n🎯 Taxa por Ticker:\n")
for _, row in ticker_stats_df.iterrows():
    print(f"{row['ticker']}: {row['acertos']}/{row['selecionado']} ({row['taxa']*100:.2f}%) - Selecionado {row['selecionado']} vezes")

print(f"\n📈 Resumo:")
print(f"  Ticker mais selecionado: {ticker_stats_df.iloc[0]['ticker']} ({ticker_stats_df.iloc[0]['selecionado']} vezes)")
max_rate_idx = ticker_stats_df['taxa'].idxmax()
print(f"  Melhor taxa: {ticker_stats_df.loc[max_rate_idx, 'ticker']} ({ticker_stats_df.loc[max_rate_idx, 'taxa']*100:.2f}%)")

In [0]:
print("="*80)
print("🔍 COMPARAÇÃO COM RESULTADOS HISTÓRICOS")
print("="*80)

# Comparação
historical_rate = 0.5794  # 57,94%
daily_holdout_rate = 0.4777  # 47,77% (247 datas diárias)
non_overlapping_rate = top1_rate_non_overlapping

print(f"\n📈 Comparação das Taxas Top 1:\n")
print(f"  1. Histórico (treino): {historical_rate*100:.2f}%")
print(f"  2. Holdout diário (247 datas): {daily_holdout_rate*100:.2f}%")
print(f"  3. Holdout não sobreposto ({total_dates_non_overlapping} datas): {non_overlapping_rate*100:.2f}%")

print(f"\n🔽 Diferenças:\n")
diff_vs_historical = (non_overlapping_rate - historical_rate) * 100
diff_vs_daily = (non_overlapping_rate - daily_holdout_rate) * 100

print(f"  Vs. Histórico: {diff_vs_historical:+.2f} pontos percentuais")
print(f"  Vs. Holdout diário: {diff_vs_daily:+.2f} pontos percentuais")

print(f"\n🎯 Análise:\n")
if non_overlapping_rate >= 0.58:
    print(f"  ✅ Meta de 58% ATINGIDA!")
elif non_overlapping_rate >= historical_rate:
    print(f"  🟡 Superior ao histórico, mas abaixo da meta de 58%")
elif non_overlapping_rate >= daily_holdout_rate:
    print(f"  🟡 Superior ao holdout diário, mas abaixo do histórico")
else:
    print(f"  ⚠️ Abaixo de todas as referências")

print(f"\n📊 Redução de Observações:\n")
reduction = (1 - total_dates_non_overlapping / 247) * 100
print(f"  Datas diárias: 247")
print(f"  Datas não sobrepostas: {total_dates_non_overlapping}")
print(f"  Redução: {reduction:.1f}%")
print(f"  Razão: {247 / total_dates_non_overlapping:.2f}x menos observações")

In [0]:
print("\n\n")
print("#" * 80)
print("#" + " " * 78 + "#")
print("#" + " " * 20 + "🏆 RELATÓRIO FINAL - AUDITORIA TOP 1" + " " * 20 + "#")
print("#" + " " * 78 + "#")
print("#" * 80)

print(f"\n\n📊 RESULTADOS - DATAS NÃO SOBREPOSTAS (ESPAÇADAS 7 PREGÕES)")
print("="*80)

print(f"\n📋 1. Número de Datas Não Sobrepostas: {total_dates_non_overlapping}")
print(f"   (Primeira: {non_overlapping_dates[0].date()}, Última: {non_overlapping_dates[-1].date()})")

print(f"\n🎯 2. Número de Acertos (Top 1 > IFIX): {top1_hits_non_overlapping}")

print(f"\n📈 3. Taxa Top 1 de Outperformance: {top1_rate_non_overlapping:.4f} ({top1_rate_non_overlapping*100:.2f}%)")

print(f"\n📊 4. Intervalo de Confiança de 95%:")
print(f"   IC 95%: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print(f"   Margem de erro: ±{z*se*100:.2f} pontos percentuais")

print(f"\n⭐ 5. Top 1 foi o Melhor FII Absoluto:")
print(f"   Número de vezes: {top1_best_hits_non_overlapping}")
print(f"   Percentual: {top1_best_rate_non_overlapping*100:.2f}%")

print(f"\n📊 6. Taxa por Ticker Selecionado:")
for _, row in ticker_stats_df.iterrows():
    print(f"   {row['ticker']}: {row['taxa']*100:.2f}% ({row['acertos']}/{row['selecionado']} - selecionado {row['selecionado']} vezes)")

print(f"\n🔍 7. Comparação com Resultados Históricos:")
print(f"   • Histórico (treino): {historical_rate*100:.2f}%")
print(f"   • Holdout diário (247 datas): {daily_holdout_rate*100:.2f}%")
print(f"   • Holdout não sobreposto ({total_dates_non_overlapping} datas): {non_overlapping_rate*100:.2f}%")
print(f"\n   Diferença vs. Histórico: {diff_vs_historical:+.2f} p.p.")
print(f"   Diferença vs. Holdout diário: {diff_vs_daily:+.2f} p.p.")

print("\n\n" + "="*80)
print("✅ ANÁLISE COMPLETA")
print("="*80)

print(f"\n📄 Resumo Executivo:")
print(f"\n  O modelo apresentou uma taxa Top 1 de {non_overlapping_rate*100:.2f}% em {total_dates_non_overlapping} datas")
print(f"  não sobrepostas (espaçadas a cada 7 pregões), com intervalo de confiança")
print(f"  de 95% entre {ci_lower*100:.2f}% e {ci_upper*100:.2f}%.")

if non_overlapping_rate >= historical_rate:
    print(f"\n  ✅ A taxa não sobreposta ({non_overlapping_rate*100:.2f}%) é SUPERIOR ao histórico ({historical_rate*100:.2f}%).")
else:
    print(f"\n  ⚠️ A taxa não sobreposta ({non_overlapping_rate*100:.2f}%) é inferior ao histórico ({historical_rate*100:.2f}%).")

if non_overlapping_rate >= daily_holdout_rate:
    print(f"  ✅ A taxa não sobreposta ({non_overlapping_rate*100:.2f}%) é SUPERIOR ao holdout diário ({daily_holdout_rate*100:.2f}%).")
else:
    print(f"  ⚠️ A taxa não sobreposta ({non_overlapping_rate*100:.2f}%) é inferior ao holdout diário ({daily_holdout_rate*100:.2f}%).")

if non_overlapping_rate >= 0.58:
    print(f"\n  🎯 Meta de 58% ATINGIDA!")
else:
    meta_diff = (0.58 - non_overlapping_rate) * 100
    print(f"\n  🔻 Meta de 58% não atingida (faltaram {meta_diff:.2f} pontos percentuais).")

print(f"\n  O Top 1 previsto foi o melhor FII absoluto em {top1_best_rate_non_overlapping*100:.2f}% das vezes.")

print("\n" + "="*80)
print("✅ RELATÓRIO CONCLUÍDO")
print("="*80)